# 11 — Model Selection, Cross-Validation, and Pipelines

This notebook practices model selection culture: train/test split, baseline, cross-validation from scratch, hyperparameter selection, and Scikit-learn pipelines.

In [ ]:
import numpy as np

## 1. Dataset

In [ ]:
rng = np.random.default_rng(42)

n = 320
X0 = rng.multivariate_normal([-1.0, -0.8], [[1.0, 0.35], [0.35, 1.0]], size=n // 2)
X1 = rng.multivariate_normal([1.1, 0.9], [[1.0, -0.25], [-0.25, 1.0]], size=n // 2)

X = np.vstack([X0, X1])
y = np.array([0] * (n // 2) + [1] * (n // 2))

X.shape, y.shape

## 2. Train/Test Split

In [ ]:
def train_test_split_numpy(X, y, test_size=0.25, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    idx = rng.permutation(n)
    test_n = int(n * test_size)
    return X[idx[test_n:]], X[idx[:test_n]], y[idx[test_n:]], y[idx[:test_n]]

X_train, X_test, y_train, y_test = train_test_split_numpy(X, y)

X_train.shape, X_test.shape

## 3. K-Fold Indices

In [ ]:
def kfold_indices(n_samples, k=5, seed=42):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(n_samples)
    folds = np.array_split(indices, k)
    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        yield train_idx, val_idx

[(len(tr), len(val)) for tr, val in kfold_indices(len(y_train), k=5)]

## 4. Simple Logistic Regression Helpers

In [ ]:
def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1 / (1 + np.exp(-z))


def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std = np.where(std == 0, 1, std)
    return (X_train - mean) / std, (X_test - mean) / std


def train_logistic_regression(X, y, lr=0.2, steps=1000, lambda_=0.0):
    X_bias = np.column_stack([np.ones(X.shape[0]), X])
    beta = np.zeros(X_bias.shape[1])
    for _ in range(steps):
        p = sigmoid(X_bias @ beta)
        gradient = (1 / len(y)) * X_bias.T @ (p - y)
        reg = np.zeros_like(beta)
        reg[1:] = 2 * lambda_ * beta[1:]
        beta -= lr * (gradient + reg)
    return beta


def predict_logistic(X, beta, threshold=0.5):
    X_bias = np.column_stack([np.ones(X.shape[0]), X])
    return (sigmoid(X_bias @ beta) >= threshold).astype(int)


def f1_score_binary(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

## 5. Cross-Validation for Hyperparameter Selection

Important: scaling is fitted inside each training fold only.

In [ ]:
def cross_validate_lambda(X, y, lambda_, k=5):
    scores = []
    for train_idx, val_idx in kfold_indices(len(y), k=k, seed=42):
        X_train_fold = X[train_idx]
        X_val_fold = X[val_idx]
        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]
        X_train_scaled, X_val_scaled = standardize_train_test(X_train_fold, X_val_fold)
        beta = train_logistic_regression(X_train_scaled, y_train_fold, lambda_=lambda_)
        pred = predict_logistic(X_val_scaled, beta)
        scores.append(f1_score_binary(y_val_fold, pred))
    return np.array(scores)

for lambda_ in [0.0, 0.001, 0.01, 0.1, 1.0]:
    scores = cross_validate_lambda(X_train, y_train, lambda_)
    print(lambda_, scores.mean(), scores.std())

## 6. Scikit-Learn Pipeline + GridSearchCV

In [ ]:
try:
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVC
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.metrics import f1_score

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC())
    ])

    param_grid = {
        'svm__kernel': ['linear', 'rbf'],
        'svm__C': [0.1, 1, 10],
        'svm__gamma': ['scale', 0.1, 1.0],
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = GridSearchCV(pipe, param_grid=param_grid, scoring='f1', cv=cv)
    search.fit(X_train, y_train)

    pred = search.predict(X_test)
    print(search.best_params_)
    print(search.best_score_)
    print(f1_score(y_test, pred))
except Exception as exc:
    print('Skipped:', exc)

## Reflection

Model selection is not just trying many models. It is a clean experimental protocol that protects the honesty of evaluation.